### Verify the December file
Rows and units in the source file, to compare with what was loaded.

In [0]:
df_dec_file = spark.read.options(header=True, inferSchema=True).csv(
    "/Volumes/fmcg/gold/parent_incremental_update/"
)
print("Rows in file: ", df_dec_file.count())
print("Units in file:", df_dec_file.agg({"sold_quantity": "sum"}).first()[0])

Rows in file:  4485
Units in file: 3398439


### Load Atlikon's December orders
Adds the parent's December data from the `parent_incremental_update` volume to `fact_orders`. COPY INTO skips files it has already loaded, so it's safe to re-run.

In [0]:
%sql
COPY INTO fmcg.gold.fact_orders
FROM (
  SELECT
    CAST(date AS DATE) AS date,
    product_code,
    customer_code,
    CAST(sold_quantity AS BIGINT) AS sold_quantity
  FROM '/Volumes/fmcg/gold/parent_incremental_update/fact_orders.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
0,0,0


### Verify fact_orders

In [0]:
print("Rows in fact_orders:", spark.table("fmcg.gold.fact_orders").count())

Rows in fact_orders: 101212
